In [3]:
from comm_research.setup.paths import PROJECT_ROOT

In [4]:
import pandas as pd
import polars as pl
import numpy as np
import scipy.stats as st

In [5]:
from comm_research.infra.lake.parquet import scan_parquet
from comm_research.infra.tools.vis import *

In [6]:
## global vars

ACTUALS_DIR = PROJECT_ROOT / "data/clean/weather/era5/t_2m/socal"  
FCAST_DIR = PROJECT_ROOT / "data/clean/weather/gefs/v12/t_2m/socal/4wk"  
GRID_ID = 0

In [7]:
def load_data(data_dir, grid_id=None):
    grid_id = GRID_ID if grid_id is None else grid_id
    paths = sorted(Path(data_dir).glob(f"*_{grid_id}.csv"))
    if not paths:
        raise FileNotFoundError(f"No CSV for grid ID {grid_id} in {data_dir}")
    if len(paths) != 1:
        raise ValueError(f"Multiple CSVs for grid ID {grid_id} in {data_dir}: {paths}")
    return (pd.read_csv(paths[0], parse_dates=["timestamp"])
            .sort_values("timestamp")
            .reset_index(drop=True))

def utc(value):
        value = pd.Timestamp(value)
        return (
            value.tz_localize("UTC")
            if value.tzinfo is None
            else value.tz_convert("UTC")
        )


In [8]:
actuals = load_data(ACTUALS_DIR)
forecasts = load_data(FCAST_DIR)

In [9]:
actuals

,grid_id,timestamp,lat,lon,2m_mean,2m_max_6h,2m_min_6h,n_samples,valid_end,sampling_hours,units,aggregation
0,0,2000-01-01 00:00:00+00:00,33.75,-121.0,285.803528,286.418701,285.173096,28,2000-01-08T00:00:00.000000+0000,6,K,mean_of_6h_instantaneous_samples
1,0,2000-01-02 00:00:00+00:00,33.75,-121.0,285.814026,286.418701,285.173096,28,2000-01-09T00:00:00.000000+0000,6,K,mean_of_6h_instantaneous_samples
2,0,2000-01-03 00:00:00+00:00,33.75,-121.0,285.816598,286.418701,285.171387,28,2000-01-10T00:00:00.000000+0000,6,K,mean_of_6h_instantaneous_samples
3,0,2000-01-04 00:00:00+00:00,33.75,-121.0,285.853315,286.418701,285.171387,28,2000-01-11T00:00:00.000000+0000,6,K,mean_of_6h_instantaneous_samples
4,0,2000-01-05 00:00:00+00:00,33.75,-121.0,285.757520,286.418701,284.912109,28,2000-01-12T00:00:00.000000+0000,6,K,mean_of_6h_instantaneous_samples
...,...,...,...,...,...,...,...,...,...,...,...,...
9727,0,2026-09-20 00:00:00+00:00,33.75,-121.0,291.305545,292.298248,290.290894,28,2026-09-27T00:00:00.000000+0000,6,K,mean_of_6h_instantaneous_samples
9728,0,2026-09-21 00:00:00+00:00,33.75,-121.0,291.317571,292.298248,290.290894,28,2026-09-28T00:00:00.000000+0000,6,K,mean_of_6h_instantaneous_samples
9729,0,2026-09-22 00:00:00+00:00,33.75,-121.0,291.314440,292.298248,290.133789,28,2026-09-29T00:00:00.000000+0000,6,K,mean_of_6h_instantaneous_samples
9730,0,2026-09-23 00:00:00+00:00,33.75,-121.0,291.349196,292.298248,290.133789,28,2026-09-30T00:00:00.000000+0000,6,K,mean_of_6h_instantaneous_samples


### Constructing Climatological Thresholds

In [10]:
def collect_historical_data(
    df, target_date, window, *, history_years=10, lead_days=26
):
    target = pd.Timestamp(target_date, tz="UTC").normalize()
    w = pd.Timedelta(window)
    ts = pd.to_datetime(df["timestamp"], utc=True)
    vals = pd.to_numeric(df["2m_mean"])
    
    eligible = (ts + pd.Timedelta(days=7)).le(target - pd.Timedelta(days=lead_days)) & vals.notna()
    
    mask = pd.Series(False, index=df.index)
    for y in range(target.year - history_years, target.year):
        try:
            center = target.replace(year=y)
        except ValueError:
            center = target.replace(year=y, day=28)
        mask |= ts.between(center - w, center + w)
        
    return vals.loc[mask & eligible].values

In [12]:
def construct_thresholds(data, buckets=5):
    # constructs array of q_i climatology thresholds 
    q_values = [(i / buckets) for i in range(1, buckets + 1)]

    thresholds = np.quantile(
        data,
        q_values,
        method="linear",
    )

    return np.append(thresholds, np.inf)

In [13]:
def construct_cdf(df, target_date, q):
    # constructs cdf vector for model fcasts
    fcast_cols = [col for col in df.columns if 'perturbed' in col]
    fcast_cols.append('control')

    fcast_values = np.array(df.loc[df['timestamp'] == target_date, fcast_cols])

    comparison_matrix = (fcast_values[:, None] < q[None, :])
    return np.mean(comparison_matrix, axis=1)

In [14]:
def construct_actuals(df, target_date, q):
    # ground truth target vector to compare against empirical CDF
    realized_era5 = df.loc[df['timestamp'] == target_date, 'actual_mean_k']
    return (q > realized_era5).astype(int)

### Constructing Model-Ensemble CDFs

### PBC Algorithm

In [15]:
def debiasPP():
    pass

In [16]:
def persistencePP():
    pass

### Isotonic Projection and PBC Corrected Forecast

In [17]:
def isotonic_proj():
    pass

In [18]:
def pbc_forecast():
    pass

### Full Runthrough

In [ ]:
TARGET_DATE = '2026-09-24'
WINDOW = '14D'

data = collect_historical_data(
    actuals,
    TARGET_DATE,
    WINDOW
)

th = construct_thresholds(
    data,
    buckets=10
)

array([], shape=(0, 11), dtype=float64)